# Train your personal coding model

Runs the full `mycoder/train/` pipeline on a free Colab T4:

1. Check GPU
2. Clone this repo + install deps
3. Get training data (upload `train.jsonl` / `val.jsonl`, or generate pairs with your GLM/DeepSeek keys)
4. QLoRA fine-tune Qwen2.5-Coder-1.5B (`finetune.py`)
5. Merge LoRA (`merge_lora.py`) and quantize to GGUF (`quantize.sh`)
6. Download `mycoder-1.5b-q4_k_m.gguf` and copy it to your server

**Runtime > Change runtime type > T4 GPU**, then *Runtime > Run all*.
A 5k-example run takes roughly 1-2 hours on a T4.

In [ ]:
# 1) GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Repo + deps  (if the repo is private, paste a token into the URL:
#    https://<TOKEN>@github.com/I-fredericks/mycoder.git)
import os
if not os.path.isdir('mycoder'):
    !git clone https://github.com/I-fredericks/mycoder.git
%cd mycoder
!pip install -q "transformers>=4.46" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.44" datasets accelerate

In [ ]:
# 3a) Upload your data files: train.jsonl and val.jsonl (from prepare_data.py)
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
print('Select data/train.jsonl and data/val.jsonl (ctrl-click both)')
up = files.upload()
for name, content in up.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
for p in ('data/train.jsonl', 'data/val.jsonl'):
    n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
    print(p, '->', n, 'examples')

In [ ]:
# 3b) OPTIONAL: generate extra pairs right here with your GLM/DeepSeek keys.
# Skip this cell if you already uploaded data.
import getpass
glm = getpass.getpass('GLM_API_KEY (enter to skip): ')
ds  = getpass.getpass('DEEPSEEK_API_KEY (enter to skip): ')
if glm or ds:
    env = {'GLM_API_KEY': glm, 'DEEPSEEK_API_KEY': ds, 'PATH': os.environ['PATH']}
    !GLM_API_KEY={glm} DEEPSEEK_API_KEY={ds} python train/generate_data.py \
        --mode auto --stack "Python,JavaScript,SQL" --num 300 --out pairs.jsonl
    !python train/prepare_data.py --pairs pairs.jsonl --out data/train.jsonl --val data/val.jsonl

In [ ]:
# 4) QLoRA fine-tune (checkpoint is saved each epoch to out/lora)
!python train/finetune.py --train data/train.jsonl --val data/val.jsonl --out out/lora

In [ ]:
# 5) Merge + quantize -> out/mycoder-1.5b-q4_k_m.gguf
!python train/merge_lora.py --lora out/lora --out out/merged
!bash train/quantize.sh out/merged mycoder-1.5b
!ls -lh out/*.gguf

In [ ]:
# 6) Download the GGUF, then from your machine:
#    scp mycoder-1.5b-q4_k_m.gguf root@SERVER:/opt/models/coder.gguf
#    sudo systemctl restart llama
from google.colab import files
files.download('out/mycoder-1.5b-q4_k_m.gguf')